In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/processed/final_data.csv')
df['RAIN_TOTAL'] = df['RAIN_TOTAL'].replace('-', 0.0)
df['RAIN_TOTAL'] = df['RAIN_TOTAL'].astype('float64')
df.info()

In [ ]:
df['RAIN_TOTAL'].isnull().sum()
df['RAIN_TOTAL'] = df['RAIN_TOTAL'].fillna(0)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
features = ['FLOOD_AREA', 'RAIN_TOTAL', 'FLOOD_GRD', 'DAM_RAIN', 'DEAD', 'RIV_GRD', 'RIV_DIS_MIN', 'RIV_DIS_GRD', 'DRAINAGE_GRD', 'PUMP_CNT']
x = df[features]

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [ ]:
kmeans = KMeans(
    n_clusters = 4,
    init = 'k-means++',
    max_iter = 300,
    random_state = 0
)

df['cluster'] = kmeans.fit_predict(x_scaled)
print(df.groupby('cluster')[features].mean())

In [ ]:
limit = df['FLOOD_AREA'].quantile(0.98)
df_refined = df[df['FLOOD_AREA'] <= limit].copy()

features = ['FLOOD_AREA', 'RAIN_TOTAL', 'FLOOD_GRD', 'DAM_RAIN', 'DEAD', 'RIV_GRD', 'RIV_DIS_MIN',  'DRAINAGE_GRD', 'PUMP_CNT']
x = df_refined[features]

x_refined = scaler.fit_transform(df_refined[features])

In [ ]:
kmeans = KMeans(
    n_clusters = 6,
    init = 'k-means++',
    max_iter = 300,
    random_state = 0
)

df_refined['cluster'] = kmeans.fit_predict(x_refined)
print(df_refined.groupby('cluster')[features].mean())

In [ ]:
df_refined[df_refined['cluster'] == 5]

In [ ]:
df_final = df_refined[df_refined['cluster'] != 5].copy()

features = ['FLOOD_AREA', 'DAM_RAIN', 'RIV_DIS_MIN', 'DRAINAGE_GRD', 'PUMP_CNT']
x = df_final[features]

x_final = scaler.fit_transform(df_final[features])

kmeans = KMeans(
    n_clusters=4,
    init='k-means++',
    random_state=0
    )

df_final['cluster'] = kmeans.fit_predict(x_final)
print(df_final.groupby('cluster')[features].mean())

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(x_final, df_final['cluster'])
print(f"실루엣 계수: {score:.4f}")